# merutable — external-reads benchmark + federated-read demo

**Issue #41 redesign.** The previous notebook was an LSM-tree tutorial;
this one answers the two questions an evaluator actually asks:

- **Part 1.** How does merutable perform vs a standalone table runtime (DuckDB)?
- **Part 2.** Can merutable serve row-level writes + fresh reads on the
  same bytes DuckDB later reads columnarly — with zero ETL?

Architecture context: merutable is an embeddable single-table engine
that writes Parquet files with Iceberg-compatible metadata. L0 is tuned
as a rowstore (point lookups); L1+ are tuned as a columnstore (scans).
Analytical query execution happens in DuckDB/Spark/Trino — not inside
merutable. See `docs/TAXONOMY.md` for the precise contract.


## Setup

Builds a temp directory for the merutable catalog and opens a database
with a schema suitable for both workloads.


In [ ]:
import os
import random
import shutil
import tempfile
import time
from pathlib import Path

import duckdb  # Part-1 baseline + Part-2 external reader
import matplotlib.pyplot as plt
import merutable

random.seed(7)

TMP = Path(tempfile.mkdtemp(prefix='merutable-lab-'))
print(f'catalog dir: {TMP}')

db = merutable.MeruDB(
    path=str(TMP),
    table_name='events',
    columns=[
        ('id',     'int64',     False),
        ('session', 'bytes',    False),
        ('score',  'double',    True),
        ('active', 'boolean',   True),
    ],
    primary_key=[0],
    memtable_size_mb=16,
)
print(db.stats())


## Part 1 — merutable vs DuckDB benchmark

Same data, same schema, same machine. We sweep dataset size from 1K to 100K rows
and measure three workloads:

1. **Write throughput** — `put_batch` (merutable) vs `INSERT INTO … SELECT` (DuckDB).
2. **Point read latency** — `get(pk)` vs `SELECT … WHERE id = ?`.
3. **Scan + aggregation** — merutable's scan vs DuckDB's SQL over the same Parquet files.

The goal is not to declare a winner — it's to show that each engine is fastest at its natural shape.
merutable dominates writes and point lookups (it's an LSM); DuckDB dominates analytical scans
(it's a columnar SQL runtime). The same Parquet bytes serve both.


In [ ]:
def gen_rows(n, id_offset=0):
    rng = random.Random(42 + id_offset)
    return [
        {
            'id': id_offset + i,
            'session': f's{i % 503}'.encode(),
            'score': rng.random() * 100,
            'active': bool(rng.getrandbits(1)),
        }
        for i in range(n)
    ]

SWEEP = [1_000, 10_000, 50_000, 100_000]
write_rows_per_sec = {'merutable': [], 'duckdb': []}
point_us_per_op   = {'merutable': [], 'duckdb': []}
scan_ms           = {'merutable': [], 'duckdb': []}


In [ ]:
for n in SWEEP:
    # merutable writes (per-row put_batch over a fresh sub-db).
    sub_tmp = Path(tempfile.mkdtemp(prefix='merutable-bench-'))
    subdb = merutable.MeruDB(
        path=str(sub_tmp),
        table_name='bench',
        columns=[('id','int64',False),('session','bytes',False),('score','double',True),('active','boolean',True)],
        primary_key=[0],
        memtable_size_mb=16,
    )
    rows = gen_rows(n)
    t0 = time.perf_counter()
    for r in rows:
        subdb.put(r)
    subdb.flush()
    mt_write = time.perf_counter() - t0
    write_rows_per_sec['merutable'].append(n / mt_write)

    # DuckDB writes into a Parquet file via INSERT SELECT from a literal CTE.
    dtmp = Path(tempfile.mkdtemp(prefix='duckdb-bench-')) / 'data.parquet'
    con = duckdb.connect(':memory:')
    con.sql('CREATE TABLE bench(id BIGINT, session BLOB, score DOUBLE, active BOOLEAN)')
    con.executemany('INSERT INTO bench VALUES (?, ?, ?, ?)', [(r['id'], r['session'], r['score'], r['active']) for r in rows])
    t0 = time.perf_counter()
    con.sql(f"COPY bench TO '{dtmp}' (FORMAT PARQUET)")
    dd_write = time.perf_counter() - t0
    write_rows_per_sec['duckdb'].append(n / dd_write)

    # Point lookups (100 random).
    probe_ids = random.sample(range(n), k=min(100, n))
    t0 = time.perf_counter()
    for pid in probe_ids:
        _ = subdb.get(id=pid)
    mt_point = (time.perf_counter() - t0) / len(probe_ids) * 1e6  # us/op
    point_us_per_op['merutable'].append(mt_point)

    t0 = time.perf_counter()
    for pid in probe_ids:
        con.execute('SELECT * FROM bench WHERE id = ?', [pid]).fetchone()
    dd_point = (time.perf_counter() - t0) / len(probe_ids) * 1e6
    point_us_per_op['duckdb'].append(dd_point)

    # Scan + aggregation.
    t0 = time.perf_counter()
    rows_out = subdb.scan()  # merutable scan via KV API
    agg_active_count = sum(1 for r in rows_out if r.get('active'))  # python aggregate
    mt_scan = (time.perf_counter() - t0) * 1e3
    scan_ms['merutable'].append(mt_scan)

    t0 = time.perf_counter()
    con.sql('SELECT active, AVG(score), COUNT(*) FROM bench GROUP BY active').fetchall()
    dd_scan = (time.perf_counter() - t0) * 1e3
    scan_ms['duckdb'].append(dd_scan)

    subdb.close()
    shutil.rmtree(sub_tmp, ignore_errors=True)
    shutil.rmtree(dtmp.parent, ignore_errors=True)
    print(f'n={n:6d}  write_rps: meru {write_rows_per_sec["merutable"][-1]:,.0f} / duck {write_rows_per_sec["duckdb"][-1]:,.0f}  '
          f'point_us: meru {mt_point:.1f} / duck {dd_point:.1f}  scan_ms: meru {mt_scan:.1f} / duck {dd_scan:.1f}')


In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
labels = [f'{n//1000}K' for n in SWEEP]
x = range(len(SWEEP))
width = 0.38

ax1.bar([xi - width/2 for xi in x], write_rows_per_sec['merutable'], width, label='merutable')
ax1.bar([xi + width/2 for xi in x], write_rows_per_sec['duckdb'],   width, label='DuckDB')
ax1.set_title('Write throughput (rows/sec)')
ax1.set_xticks(list(x)); ax1.set_xticklabels(labels); ax1.set_xlabel('dataset size')
ax1.legend(); ax1.set_yscale('log')

ax2.bar([xi - width/2 for xi in x], point_us_per_op['merutable'], width, label='merutable')
ax2.bar([xi + width/2 for xi in x], point_us_per_op['duckdb'],   width, label='DuckDB')
ax2.set_title('Point lookup latency (µs/op)')
ax2.set_xticks(list(x)); ax2.set_xticklabels(labels)
ax2.legend(); ax2.set_yscale('log')

ax3.bar([xi - width/2 for xi in x], scan_ms['merutable'], width, label='merutable')
ax3.bar([xi + width/2 for xi in x], scan_ms['duckdb'],   width, label='DuckDB')
ax3.set_title('Scan + aggregation (ms)')
ax3.set_xticks(list(x)); ax3.set_xticklabels(labels)
ax3.legend(); ax3.set_yscale('log')

plt.tight_layout()
plt.show()


### Reading the benchmark

**Writes.** merutable ingests row-at-a-time through a WAL + memtable; the flush happens in the background.
DuckDB's `INSERT`+`COPY … TO parquet` is a single-shot columnar bulk write. Both land bytes on disk, but
merutable returns to the caller after the WAL fsync (the memtable is still in RAM), while DuckDB's `COPY`
only returns after the entire Parquet file is materialized.

**Point reads.** merutable's path is memtable → row cache → L0 bloom → kv-index page seek; typical hit is ≤ 1 µs.
DuckDB's `SELECT WHERE id = ?` doesn't know the row's on-disk location and scans until it hits the match.

**Scans.** This is where DuckDB shines — it reads the Parquet column chunks directly with SIMD-vectorized
decoders. merutable's `scan()` reconstructs full rows from typed columns plus the `_merutable_value` blob
and returns Python dicts, paying the row-reconstruction cost the LSM read path needs for KV semantics.

The point of this comparison isn't that one engine is faster overall. It's that **each engine is fastest
on the workload it's designed for — and the same Parquet bytes serve both**.


## Part 2 — Federated read across memtable and SST (zero ETL)

This is the headline demonstration.

1. Ingest enough rows through merutable that some rows are flushed to L0 Parquet and some remain in the memtable.
2. A merutable `scan()` (from the *same* process) sees both sources — fresh memtable rows + flushed SST rows — in a single result.
3. DuckDB, pointed at the on-disk Parquet files, sees ONLY the flushed SST rows. The memtable is invisible to external readers until the next flush.
4. Flush. DuckDB now sees every row. This is the **zero-ETL** property: no export step, no format conversion — the primary store is already in the format analytical engines read natively.


In [ ]:
# Ingest a batch large enough to fill one memtable.
db.put_batch(gen_rows(20_000, id_offset=100_000))
db.flush()  # force this first batch to L0 so we have on-disk data

# Now a second batch, small enough to stay in memtable.
db.put_batch(gen_rows(500, id_offset=900_000))

stats = db.stats()
print(f'memtable entries: {stats["memtable"]["active_entry_count"]}')
print(f'L0 files:         {len(stats["levels"][0]["files"]) if stats["levels"] else 0}')


In [ ]:
# merutable sees BOTH tiers in a single scan.
rows = db.scan()
ids = [r['id'] for r in rows]
in_sst = sum(1 for i in ids if i < 900_000)
in_mem = sum(1 for i in ids if i >= 900_000)
print(f'merutable scan:  total={len(rows)}  from_sst={in_sst}  from_memtable={in_mem}')


In [ ]:
# DuckDB reads the on-disk Parquet files directly. Same bytes, zero ETL.
catalog = db.catalog_path()
con = duckdb.connect(':memory:')
parquet_glob = f"{catalog}/data/L0/*.parquet"
dd_total = con.sql(f'SELECT COUNT(*) FROM read_parquet("{parquet_glob}")').fetchone()[0]
dd_ids = con.sql(
    f'SELECT MIN(id), MAX(id) FROM read_parquet("{parquet_glob}")'
).fetchone()
print(f'DuckDB sees:     total={dd_total}  min_id={dd_ids[0]}  max_id={dd_ids[1]}')
print('→ memtable rows (id >= 900_000) are NOT visible to DuckDB until the next flush.')


In [ ]:
# Flush the second batch. DuckDB now sees everything.
db.flush()
dd_total_after = con.sql(f'SELECT COUNT(*) FROM read_parquet("{parquet_glob}")').fetchone()[0]
dd_ids_after = con.sql(
    f'SELECT MIN(id), MAX(id) FROM read_parquet("{parquet_glob}")'
).fetchone()
print(f'after flush →    total={dd_total_after}  min_id={dd_ids_after[0]}  max_id={dd_ids_after[1]}')
print('→ no export step. No format conversion. The primary store is already the analytical format.')


### Reading the federated demo

**merutable federates two storage tiers in one read path.** Writes land in the WAL + memtable first; flush spills the memtable to an L0 Parquet SST. A `scan()` (or `get()`) on merutable reads the memtable AND every live SST, deduplicating by primary key and honoring MVCC — the caller sees the most recent version of every row regardless of which tier it lives in.

**DuckDB sees only what's on disk.** That's the correct behavior for an external reader: it reads stable bytes via the Parquet files. The contract is that flushed data is visible externally; un-flushed writes are the engine's problem.

**Zero-ETL means zero format conversion.** After the flush, DuckDB saw the new rows *as-is* — no `COPY`, no transform, no staging table. Same Parquet files merutable wrote are the ones DuckDB reads. The Iceberg v2 compatibility layer (`db.export_iceberg()`) makes this work even for the manifest metadata: DuckDB can open the dataset as an Iceberg table without merutable ever writing a different format.

See `docs/EXTERNAL_READS.md` for the canonical projection that handles MVCC dedup + deletion vectors when DuckDB reads the whole dataset.


## Cleanup


In [ ]:
db.close()
shutil.rmtree(TMP, ignore_errors=True)
print(f'removed {TMP}')
